# 🔬 Eksperimen 2: Repeated LOPO-CV & Uji Konsistensi (Hingga 10x Perulangan)
**Kelompok Peneliti**: Kelompok A01 — RPL A (Magister Teknik Informatika ITS)  
**Dataset Acuan**: `cleaned_test_features.csv` (Group-Aware Median Imputation, 0 Missing Values, 24 Proyek Utuh)  
**Tujuan Eksperimen**: Menguji kestabilan, variansi stokastik, dan konsistensi empiris dari 24 pipeline faktorial (S1/S2/S3 × B1/B2/B3/B4 × M1/M2) lintas **10 independent random seeds**.  
**Daftar Seeds Baku (10x)**: `[42, 100, 2024, 777, 999, 123, 456, 789, 2025, 2026]`  
**Protokol Validasi**: Leave-One-Project-Out Cross-Validation (Zero Data Leakage).

## 1. Setup Lingkungan & Import Modul Eksperimen

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Hubungkan ke direktori src
src_dir = str(Path("../src").resolve())
if src_dir not in sys.path:
    sys.path.append(src_dir)

from data_loader import load_cleaned_dataset
from pipeline_builder import get_pipeline_specs
from repeated_lopo_runner import run_repeated_lopo_experiments, DEFAULT_SEEDS_10
from consistency_analyzer import analyze_pipeline_consistency

print(f"[OK] 10 Baku Random Seeds: {DEFAULT_SEEDS_10}")
all_specs = get_pipeline_specs()
print(f"[OK] Total Pipeline Specs Terdefinisi: {len(all_specs)} Pipelines")

## 2. Ingestion & Verifikasi Dataset Terimputasi Bersih (Cleaned)

In [ ]:
df, features, projects = load_cleaned_dataset()
print(f"[STATUS] Total Baris Data : {len(df):,}")
print(f"[STATUS] Total Proyek     : {len(projects)} repositori")
print(f"[STATUS] Total Fitur      : {len(features)} fitur")
print(f"[STATUS] Populasi Flaky   : {df['flaky'].sum():,} ({df['flaky'].mean()*100:.2f}%)")

## 3. Eksekusi Uji Konsistensi (Repeated LOPO-CV)
Pilihan mode eksekusi:
- **Pilihan A: Smoke Test Cepat**: Verifikasi 2 pipelines pada 2 seeds & 2 folds.
- **Pilihan B: Top 10 Pipelines (Rekomendasi)**: Menguji 10 pipeline teratas lintas 10 random seeds.
- **Pilihan C: Full 24 Pipelines**: Menguji seluruh 24 pipeline faktorial lintas 10 seeds.

In [ ]:
# Pilihan A: Mode Cepat / Smoke Test (2 Pipelines, 2 Seeds)
results_quick = run_repeated_lopo_experiments(
    pipelines=[all_specs[0], all_specs[12]],
    seeds=DEFAULT_SEEDS_10[:2],
    output_dir="../results_quick_test",
    use_gpu=True,
    quick_test=True
)
display(results_quick["all_runs"].head())

In [ ]:
# Pilihan B: Eksekusi Top 10 Pipelines Eksperimen 2 Lintas 10 Random Seeds
top_pipeline_ids = [
    "P13_S2_B3_M1", "P21_S3_B3_M1", "P05_S1_B3_M1", "P06_S1_B3_M2",
    "P14_S2_B3_M2", "P22_S3_B3_M2", "P12_S2_B2_M2", "P08_S1_B4_M2",
    "P16_S2_B4_M2", "P24_S3_B4_M2"
]
selected_pipelines = [p for p in all_specs if p["pipeline_id"] in top_pipeline_ids]

print(f"Memulai repeated evaluation untuk {len(selected_pipelines)} top pipelines lintas 10 seeds...")
results_10x = run_repeated_lopo_experiments(
    pipelines=selected_pipelines,
    seeds=DEFAULT_SEEDS_10, # 10 Random Seeds
    output_dir="../results",
    use_gpu=True
)

## 4. Analisis Konsistensi & Tabel Stabilitas Empiris
Menghitung Mean ($\mu$), Standar Deviasi ($\sigma$), Coefficient of Variation ($CV = \frac{\sigma}{\mu} \times 100\%$), serta rentang nilai (Min-Max) dan 95% Confidence Interval.

In [ ]:
df_cons = analyze_pipeline_consistency(
    all_runs_csv="../results/all_seed_pipeline_runs.csv",
    output_dir="../results"
)

cols_show = [
    "Pipeline_ID", "Scaling", "Imbalance", "Model", "Num_Seeds",
    "Mean_F1_Mean", "Mean_F1_Std", "Mean_F1_CV_%", "Mean_F1_CI95",
    "Mean_Recall_Mean", "Mean_Precision_Mean", "Mean_PR_AUC_Mean"
]
display(df_cons[[c for c in cols_show if c in df_cons.columns]].head(10))

## 5. Visualisasi Kestabilan Lintas 10 Repetisi

In [ ]:
df_runs = pd.read_csv("../results/all_seed_pipeline_runs.csv")

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_runs, x="Pipeline_ID", y="Mean_F1", hue="Model")
plt.xticks(rotation=45, ha="right")
plt.title("Distribusi F1-Score Lintas 10 Repetisi Random Seeds (Uji Konsistensi)", fontsize=14, fontweight="bold")
plt.ylabel("Mean F1-Score")
plt.xlabel("Pipeline ID")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()